In [ ]:
# GOOD CELL
# Install dependencies if needed:
# !pip install ipywidgets matplotlib pandas ipydatagrid
%matplotlib widget
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from ipydatagrid import DataGrid
import pandas as pd

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv(r'..\Code\159_planets_all_columns_with_AH_2.csv')


# -----------------------------
# SETTINGS
# -----------------------------
plot_pairs = [('pl_Teq', 'A/H'), ('pl_Teq', 'pl_tsm'), ('pl_Teq', 'Feature Height (ppm)')]
plot_index = 0
selected_planets = pd.DataFrame(columns=df.columns)

# -----------------------------
# WIDGETS
# -----------------------------
x_dropdown = widgets.Dropdown(description='X-axis:')
y_dropdown = widgets.Dropdown(description='Y-axis:')
input_box = widgets.Text(description='Search:', placeholder='e.g., 800')
left_btn = widgets.Button(description='<')
right_btn = widgets.Button(description='>')
info_out = widgets.Output()
plot_out = widgets.Output()
clear_btn = widgets.Button(description='Clear Selection', button_style='warning')


# Scrollable table
grid = DataGrid(df, selection_mode='row', layout={'height': '300px'})
display(grid)

# -----------------------------
# FIGURE
# -----------------------------
#fig, ax = plt.subplots(figsize=(6,4))

# -----------------------------
# HELPER FUNCTIONS
# -----------------------------
def update_dropdowns():
    numeric_cols = list(df.select_dtypes(include='number').columns)
    x_dropdown.options = numeric_cols
    y_dropdown.options = numeric_cols
    
    # Safe defaults
    if x_dropdown.value is None:
        x_dropdown.value = 'pl_Teq' if 'pl_Teq' in numeric_cols else numeric_cols[0]
    if y_dropdown.value is None:
        y_dropdown.value = 'A/H' if 'A/H' in numeric_cols else numeric_cols[1] if len(numeric_cols) > 1 else numeric_cols[0]



def plot_scatter(x_col, y_col):
    if x_col not in df.columns or y_col not in df.columns:
        return

    with plot_out:
        plot_out.clear_output(wait=True)
        fig, ax = plt.subplots()
        ax.scatter(df[x_col], df[y_col], s=20, alpha=0.5, picker=True)  # picker=True here
        if not selected_planets.empty:
            ax.scatter(selected_planets[x_col], selected_planets[y_col], s=60, alpha=0.7, fc = 'None', ec='red', marker='o')
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.set_title("Planet comparison")
        ax.grid(True)
        fig.canvas.mpl_connect('pick_event', onpick_multi)  # connect here, to THIS fig
        plt.show()

# -----------------------------
# CALLBACKS
# -----------------------------
def onpick_multi(event):
    global selected_planets
    if event.mouseevent.button != 1:  # left click only
        return
    ind = event.ind[0]
    planet = df.iloc[ind]
    
    # toggle selection
    if planet['pl_name'] in selected_planets['pl_name'].values:
        selected_planets = selected_planets[selected_planets['pl_name'] != planet['pl_name']]
    else:
        selected_planets = pd.concat([selected_planets, pd.DataFrame([planet])], ignore_index=True)
    
    # redraw
    plot_scatter(x_dropdown.value, y_dropdown.value)
    
    # info output
    with info_out:
        clear_output(wait=True)
        display(f"Selected planets ({len(selected_planets)}):")
        if not selected_planets.empty:
            display(selected_planets[['pl_name', x_dropdown.value, y_dropdown.value]])

def on_input_change(change):
    global selected_planets
    if change['type'] == 'change' and change['name'] == 'value' and change['new']:
        try:
            val = float(change['new'])
        except:
            with info_out:
                clear_output(wait=True)
                display("Enter a numeric value")
            return
        
        # search within 5% of value
        matches = df[abs(df[x_dropdown.value] - val) < (0.05 * val)]
        if matches.empty:
            with info_out:
                clear_output(wait=True)
                display("No planets found in range")
            return
        
        # add matches
        for _, planet in matches.iterrows():
            if planet['pl_name'] not in selected_planets['pl_name'].values:
                selected_planets = pd.concat([selected_planets, pd.DataFrame([planet])], ignore_index=True)
        
        plot_scatter(x_dropdown.value, y_dropdown.value)
        
        with info_out:
            clear_output(wait=True)
            display(f"Selected planets ({len(selected_planets)}):")
            display(selected_planets[['pl_name', x_dropdown.value, y_dropdown.value]])

def on_left_click(b):
    global plot_index
    plot_index = (plot_index - 1) % len(plot_pairs)
    x_dropdown.value, y_dropdown.value = plot_pairs[plot_index]
    plot_scatter(x_dropdown.value, y_dropdown.value)

def on_right_click(b):
    global plot_index
    plot_index = (plot_index + 1) % len(plot_pairs)
    x_dropdown.value, y_dropdown.value = plot_pairs[plot_index]
    plot_scatter(x_dropdown.value, y_dropdown.value)


def on_clear_click(b):
    global selected_planets
    selected_planets = pd.DataFrame(columns=df.columns)
    
    plot_scatter(x_dropdown.value, y_dropdown.value)
    
    with info_out:
        clear_output(wait=True)
        display("Selection cleared.")

# -----------------------------
# CONNECT CALLBACKS
# -----------------------------

update_dropdowns()



input_box.observe(on_input_change)
left_btn.on_click(on_left_click)
right_btn.on_click(on_right_click)
clear_btn.on_click(on_clear_click)

def on_axis_change(change):
    plot_scatter(x_dropdown.value, y_dropdown.value)

x_dropdown.observe(on_axis_change, names='value')
y_dropdown.observe(on_axis_change, names='value')



# -----------------------------
# LAYOUT
# -----------------------------
control_box = widgets.HBox([left_btn, right_btn, x_dropdown, y_dropdown, input_box, clear_btn])
display(control_box, info_out, plot_out)
# -----------------------------
# INITIAL PLOT

# -----------------------------
update_dropdowns()
plot_scatter(x_dropdown.value, y_dropdown.value)
#plt.show()


DataGrid(auto_fit_params={'area': 'all', 'padding': 30, 'numCols': None}, corner_renderer=None, default_render…

Output()

Output()

In [ ]:
# TEST CELL
# !pip install ipywidgets matplotlib pandas ipydatagrid
%matplotlib widget
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from ipydatagrid import DataGrid
import pandas as pd

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv(r'..\Code\159_planets_all_columns_with_AH_2.csv')
# df = pd.read_csv(r'..\Code\Data\results\results_2026_02_26__15_50.csv')
# df.rename(columns={df.columns[0]: "pl_name"}, inplace=True)

# -----------------------------
# SETTINGS
# -----------------------------
plot_pairs = [('pl_Teq', 'A/H'), ('pl_Teq', 'pl_tsm'), ('pl_Teq', 'Feature Height (ppm)')]
plot_index = 0
selected_planets = pd.DataFrame(columns=df.columns)

# -----------------------------
# WIDGETS
# -----------------------------
x_dropdown = widgets.Dropdown(description='X-axis:')
y_dropdown = widgets.Dropdown(description='Y-axis:')
input_box = widgets.Text(description='Search:', placeholder='e.g., 800')
left_btn = widgets.Button(description='<')
right_btn = widgets.Button(description='>')
info_out = widgets.Output()
plot_out = widgets.Output()
clear_btn = widgets.Button(description='Clear Selection', button_style='warning')


# Scrollable table
grid = DataGrid(df, selection_mode='row', layout={'height': '300px'})
display(grid)

# -----------------------------
# FIGURE
# -----------------------------
#fig, ax = plt.subplots(figsize=(6,4))

# -----------------------------
# HELPER FUNCTIONS
# -----------------------------
def update_dropdowns():
    numeric_cols = list(df.select_dtypes(include='number').columns)
    x_dropdown.options = numeric_cols
    y_dropdown.options = numeric_cols
    
    # Safe defaults
    if x_dropdown.value is None:
        x_dropdown.value = 'pl_Teq' if 'pl_Teq' in numeric_cols else numeric_cols[0]
    if y_dropdown.value is None:
        y_dropdown.value = 'A/H' if 'A/H' in numeric_cols else numeric_cols[1] if len(numeric_cols) > 1 else numeric_cols[0]



def plot_scatter(x_col, y_col):
    if x_col not in df.columns or y_col not in df.columns:
        return

    with plot_out:
        plot_out.clear_output(wait=True)
        fig, ax = plt.subplots()
        ax.scatter(df[x_col], df[y_col], s=20, alpha=0.5, picker=True)  # picker=True here
        if not selected_planets.empty:
            ax.scatter(selected_planets[x_col], selected_planets[y_col], s=60, alpha=0.7, fc = 'None', ec='red', marker='o')
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.set_title("Planet comparison")
        ax.grid(True)
        fig.canvas.mpl_connect('pick_event', onpick_multi)  # connect here, to THIS fig
        plt.show()

# -----------------------------
# CALLBACKS
# -----------------------------
def onpick_multi(event):
    global selected_planets
    if event.mouseevent.button != 1:  # left click only
        return
    ind = event.ind[0]
    planet = df.iloc[ind]
    
    # toggle selection
    if planet['pl_name'] in selected_planets['pl_name'].values:
        selected_planets = selected_planets[selected_planets['pl_name'] != planet['pl_name']]
    else:
        selected_planets = pd.concat([selected_planets, pd.DataFrame([planet])], ignore_index=True)
    
    # redraw
    plot_scatter(x_dropdown.value, y_dropdown.value)
    
    # info output
    with info_out:
        clear_output(wait=True)
        display(f"Selected planets ({len(selected_planets)}):")
        if not selected_planets.empty:
            display(selected_planets[['pl_name', x_dropdown.value, y_dropdown.value]])

def on_input_change(change):
    global selected_planets
    
    if change['type'] != 'change' or change['name'] != 'value':
        return
        
    query = change['new']
    
    if not query:
        return
    
    # -----------------------
    # TEXT SEARCH (planet name)
    # -----------------------
    if not query.replace('.', '', 1).isdigit():
        
        matches = df[df['pl_name'].str.contains(query, case=False, na=False)]
        
        if matches.empty:
            with info_out:
                clear_output(wait=True)
                display("No planet name match")
            return
        
        for _, planet in matches.iterrows():
            if planet['pl_name'] not in selected_planets['pl_name'].values:
                selected_planets = pd.concat(
                    [selected_planets, pd.DataFrame([planet])],
                    ignore_index=True
                )
    
    # -----------------------
    # NUMERIC SEARCH (your original behaviour)
    # -----------------------
    else:
        val = float(query)
        matches = df[abs(df[x_dropdown.value] - val) < (0.05 * val)]
        
        if matches.empty:
            with info_out:
                clear_output(wait=True)
                display("No planets found in range")
            return
        
        for _, planet in matches.iterrows():
            if planet['pl_name'] not in selected_planets['pl_name'].values:
                selected_planets = pd.concat(
                    [selected_planets, pd.DataFrame([planet])],
                    ignore_index=True
                )
    
    # redraw plot
    plot_scatter(x_dropdown.value, y_dropdown.value)
    
    with info_out:
        clear_output(wait=True)
        display(f"Selected planets ({len(selected_planets)}):")
        display(selected_planets[['pl_name', x_dropdown.value, y_dropdown.value]])

def on_left_click(b):
    global plot_index
    plot_index = (plot_index - 1) % len(plot_pairs)
    x_dropdown.value, y_dropdown.value = plot_pairs[plot_index]
    plot_scatter(x_dropdown.value, y_dropdown.value)

def on_right_click(b):
    global plot_index
    plot_index = (plot_index + 1) % len(plot_pairs)
    x_dropdown.value, y_dropdown.value = plot_pairs[plot_index]
    plot_scatter(x_dropdown.value, y_dropdown.value)


def on_clear_click(b):
    global selected_planets
    selected_planets = pd.DataFrame(columns=df.columns)
    
    plot_scatter(x_dropdown.value, y_dropdown.value)
    
    with info_out:
        clear_output(wait=True)
        display("Selection cleared.")

# -----------------------------
# CONNECT CALLBACKS
# -----------------------------

update_dropdowns()



input_box.observe(on_input_change)
left_btn.on_click(on_left_click)
right_btn.on_click(on_right_click)
clear_btn.on_click(on_clear_click)

def on_axis_change(change):
    plot_scatter(x_dropdown.value, y_dropdown.value)

x_dropdown.observe(on_axis_change, names='value')
y_dropdown.observe(on_axis_change, names='value')



# -----------------------------
# LAYOUT
# -----------------------------
control_box = widgets.HBox([left_btn, right_btn, x_dropdown, y_dropdown, input_box, clear_btn])
display(control_box, info_out, plot_out)
# -----------------------------
# INITIAL PLOT

# -----------------------------
update_dropdowns()
plot_scatter(x_dropdown.value, y_dropdown.value)
#plt.show()


DataGrid(auto_fit_params={'area': 'all', 'padding': 30, 'numCols': None}, corner_renderer=None, default_render…

Output()

Output()

In [ ]:
#export selection to csv wiht all columns other than discovery method cos has text in.

from datetime import datetime
now = datetime.now()

# remove extra columns with words
selected_planets = selected_planets.copy()
selected_planets.insert(2, "no_of_transits", 1)
selected_planets_dropped = selected_planets.drop(columns=['discoverymethod'])


pcount = len(selected_planets_dropped)
# Save full table
output_file = f"{pcount}_planet_list_{now.strftime('%d.%m_%H-%M')}.csv" # change file name 
selected_planets_dropped.to_csv(output_file, index=False)
print(f"Saved {len(selected_planets_dropped)} planets to {output_file}")



Saved 3 planets to 3_planet_list_26.02_16-41.csv


In [ ]:
manual_list = [] # add list of names 'name', 'name'

manual_list_planets = df[df['pl_name'].isin(manual_list)]

manual_list_planets = manual_list_planets.copy()
manual_list_planets.insert(2, "no_of_transits", 1)
manual_list_planets_dropped = manual_list_planets.drop(columns=['discoverymethod'])



pcount = len(manual_list_planets)
# Save full table
output_file = f"{pcount}_planet_list_{now.strftime('%d.%m_%H-%M')}.csv" # change file name 
manual_list_planets_dropped.to_csv(output_file, index=False)
print(f"Saved {len(manual_list_planets_dropped)} planets to {output_file}")


In [6]:
selected_planets['pl_name']

0       K2-138 f
1     TOI-1064 c
2    Kepler-11 f
Name: pl_name, dtype: object